In [1]:
!pip install kaggle

In [2]:
# configuring the path of Kaggle.json file
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [3]:
!kaggle competitions download -c cifar-10

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 4, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.12/dist-packages/kaggle/__init__.py", line 6, in <module>
    api.authenticate()
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 434, in authenticate
    raise IOError('Could not find {}. Make sure it\'s located in'
OSError: Could not find kaggle.json. Make sure it's located in /root/.kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/


In [4]:
!ls

sample_data


In [5]:
from zipfile import ZipFile
dataset = '/content/cifar-10.zip'

with ZipFile(dataset, 'r') as zip:
  zip.extractall()
  print("dataset is extracted")

FileNotFoundError: [Errno 2] No such file or directory: '/content/cifar-10.zip'

In [6]:
!ls

In [7]:
!pip install py7zr

In [8]:
import py7zr

archive = py7zr.SevenZipFile('/content/train.7z', mode='r')
archive.extractall()
archive.close()

In [9]:
!ls

In [10]:
# Importing the dependences
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [11]:
filenames = os.listdir('/content/train')
type(filenames)

In [12]:
print(filenames[0:5]) #head
print(filenames[-5:]) #tail

In [13]:
len(filenames)

##Label Processing

In [14]:
labels_df = pd.read_csv('/content/trainLabels.csv')

In [15]:
labels_df.head()

In [16]:
labels_df.shape

In [17]:
labels_df['label'].value_counts()

In [18]:
labels_dictonary = {'airplane':0, 'automobile':1, 'bird':2, 'cat':3, 'deer':4, 'dog':5, 'frog':6, 'horse':7, 'ship':8, 'truck':9}

In [19]:
labels = [labels_dictonary[i] for i in labels_df['label']]
print(labels)

In [20]:
type(labels)

In [21]:
print(labels[0:5])
labels_df.head()

In [22]:
print(labels_dictonary)

In [23]:
print(labels[0:5])

In [24]:
labels_df.head()

In [25]:
# Displaying one image
import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread('/content/train/36670.png')
cv2_imshow(img)

In [26]:
id_list = list(labels_df['id'])

In [27]:
print(id_list[0:5])
print(id_list[-5:])

## Image Processing

In [28]:
#convert images to numpy array
train_data_folder = '/content/train/'

data = []

for id in id_list:
  image = Image.open(train_data_folder + str(id) + '.png')
  image = np.array(image)
  data.append(image)

In [29]:
type(data)

In [30]:
len(data)

In [31]:
type(data[0])

In [32]:
print(data[0])

In [33]:
data[0].shape

In [34]:
# convert image list and label list to numpy array
X = np.array(data)
Y = np.array(labels)

In [35]:
print(X.shape, Y.shape)

## Train Test split

In [36]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

In [37]:
print(X.shape, X_train.shape, X_test.shape)

In [38]:
print(Y.shape, Y_train.shape, Y_test.shape)

###Scaling increase the performance of Neural Network, all values from 0 to 255 should be scale down to 0 to 1.

In [39]:
X_traint_scale = X_train/255
X_test_scale = X_test/255

In [40]:
X_traint_scale[0]

##Building the Neural Network

In [41]:
import tensorflow as tf
from tensorflow import keras

In [42]:
num_of_classes  = 10

#setting up layers of Neural Network

model = keras.Sequential([
    keras.layers.Flatten(input_shape=(32,32,3)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(num_of_classes, activation="softmax")
])

In [43]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['acc']
)

In [44]:
model.fit(X_traint_scale, Y_train, validation_split = 0.1, epochs=10)

### We will use pretrained model for getting better accuracy
**ResNet50**

In [45]:
from tensorflow.keras import Sequential, layers, models
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Model
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras import optimizers

In [46]:
convolutional_base = ResNet50(weights='imagenet', include_top=False, input_shape=(256,256,3))
convolutional_base.summary()

In [47]:
num_of_classes = 10

model = models.Sequential()
model.add(layers.UpSampling2D((2,2)))
model.add(layers.UpSampling2D((2,2)))
model.add(layers.UpSampling2D((2,2)))
model.add(convolutional_base)
model.add(layers.Flatten())
model.add(layers.BatchNormalization())
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.BatchNormalization())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.BatchNormalization())
model.add(layers.Dense(num_of_classes, activation='softmax'))

In [48]:
model.compile(
    optimizer = optimizers.RMSprop(learning_rate=2e-5),
    loss = 'sparse_categorical_crossentropy',
    metrics = ['acc']
)

In [ ]:
history = model.fit(X_traint_scale, Y_train, validation_split=0.1, epochs=10)

Epoch 1/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 483s 379ms/step - acc: 0.3220 - loss: 2.0788 - val_acc: 0.7630 - val_loss: 0.8716
Epoch 2/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 426s 379ms/step - acc: 0.6853 - loss: 1.0365 - val_acc: 0.8932 - val_loss: 0.4795
Epoch 3/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 427s 379ms/step - acc: 0.8138 - loss: 0.7239 - val_acc: 0.9212 - val_loss: 0.3394
Epoch 4/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 426s 379ms/step - acc: 0.8721 - loss: 0.5611 - val_acc: 0.9298 - val_loss: 0.2759
Epoch 5/10
 102/1125 ━━━━━━━━━━━━━━━━━━━━ 6:15 367ms/step - acc: 0.9071 - loss: 0.4430

In [ ]:
h = history

# plot the loss value
plt.plot(h.history['loss'], label='train loss')
plt.plot(h.history['val_loss'], label='validation loss')
plt.legend()
plt.show()

# plot the accuracy value
plt.plot(h.history['acc'], label='train accuracy')
plt.plot(h.history['val_acc'], label='validation accuracy')
plt.legend()
plt.show()